# 06 — Encapsulation and Scope

Encapsulation is the core OOP principle of bundling data and methods together, and controlling access to them. Instead of exposing raw data that anyone can modify, a class hides its internals behind a public interface. The class guarantees that its data is always valid — callers can't corrupt it because they can't reach it directly. This notebook covers class design, access modifiers, the `this` pointer, const methods, and getters/setters.


## class vs struct in C++

In C, `struct` is just a plain collection of data fields with no methods. In C++, both `struct` and `class` can have methods, constructors, and access modifiers. The **only** difference is the default access level:

- `struct` — members are **public** by default
- `class` — members are **private** by default

**Convention:** Use `struct` for plain data (like a C struct), use `class` for objects with behavior and invariants.


In [ ]:
#include <iostream>
#include <string>

// struct: plain data, everything public by default
struct Point {
    double x;  // public
    double y;  // public
};

// class: behavior + invariants, private by default
class Rectangle {
    double width;   // private — cannot be accessed from outside
    double height;  // private
public:
    void setDimensions(double w, double h) {
        width = w;
        height = h;
    }
    double area() const {
        return width * height;
    }
};

Point p;  // struct: access members directly
p.x = 3.0;
p.y = 4.0;
std::cout << "Point: (" << p.x << ", " << p.y << ")" << std::endl;

Rectangle rect;
rect.setDimensions(5.0, 3.0);
std::cout << "Rectangle area: " << rect.area() << std::endl;
// Expected output:
// Point: (3, 4)
// Rectangle area: 15

## Access Modifiers

C++ has three access specifiers:

- **`public`** — accessible from anywhere
- **`private`** — accessible only from within the class itself (and friend classes)
- **`protected`** — like private, but also accessible from derived (child) classes (covered in the inheritance notebook)

You can have multiple `public:` / `private:` sections in a class — they're just labels that change the access mode for everything that follows.


In [ ]:
#include <iostream>
#include <string>

class Employee {
public:
    std::string name;   // anyone can read/write this

    void introduce() const {
        std::cout << "Hi, I'm " << name
                  << ", ID=" << employeeId
                  << ", salary=" << salary << std::endl;
    }

protected:
    int employeeId;    // accessible to subclasses

private:
    double salary;     // strictly internal — even subclasses can't see this

public:
    // We can re-open public section anywhere
    void setSalary(double s) { salary = s; }
    void setId(int id)       { employeeId = id; }
};

Employee emp;
emp.name = "Alice";   // OK: public
emp.setId(1001);      // OK: public method
emp.setSalary(75000); // OK: public method
// emp.salary = 999;  // ERROR: private — would not compile
emp.introduce();
// Expected output:
// Hi, I'm Alice, ID=1001, salary=75000

## A Basic Class

Let's design a `BankAccount` class that demonstrates encapsulation properly. The balance is private — the only way to change it is through controlled methods.


In [ ]:
#include <iostream>
#include <string>

class BankAccount {
private:
    std::string owner;
    double balance;

public:
    void init(std::string ownerName, double initialBalance) {
        owner = ownerName;
        balance = initialBalance;
    }

    void deposit(double amount) {
        if (amount > 0) {
            balance += amount;
            std::cout << "Deposited $" << amount
                      << ". New balance: $" << balance << std::endl;
        } else {
            std::cout << "Invalid deposit amount." << std::endl;
        }
    }

    void withdraw(double amount) {
        if (amount <= 0) {
            std::cout << "Invalid amount." << std::endl;
        } else if (amount > balance) {
            std::cout << "Insufficient funds." << std::endl;
        } else {
            balance -= amount;
            std::cout << "Withdrew $" << amount
                      << ". New balance: $" << balance << std::endl;
        }
    }

    double getBalance() const {
        return balance;
    }

    void printStatement() const {
        std::cout << "Account owner: " << owner
                  << " | Balance: $" << balance << std::endl;
    }
};

BankAccount acc;
acc.init("Alice", 500.0);
acc.printStatement();
acc.deposit(200.0);
acc.withdraw(100.0);
acc.withdraw(1000.0);  // should fail
acc.printStatement();
// Expected output:
// Account owner: Alice | Balance: $500
// Deposited $200. New balance: $700
// Withdrew $100. New balance: $600
// Insufficient funds.
// Account owner: Alice | Balance: $600

## Trying to Access Private Members

If you try to access a private member from outside the class, the compiler rejects it. This is a compile-time error — not a runtime crash.

```cpp
BankAccount acc;
acc.balance = 1000000;  // ERROR: 'double BankAccount::balance' is private within this context
```

The compiler message tells you exactly which member and why. This is the power of encapsulation: **impossible-to-compile violations**. No runtime debugging needed.


## Getters and Setters

A **getter** reads a private value (usually `const`). A **setter** writes it, with optional validation. Why not just make the field public?

- **Validation** — the setter can reject invalid values
- **Invariants** — keep the object in a consistent state
- **Flexibility** — change internal representation later without breaking callers
- **Read-only fields** — provide a getter but no setter


In [ ]:
#include <iostream>
#include <string>

class Person {
private:
    std::string name;
    int age;

public:
    // Getter — const: promises not to modify the object
    std::string getName() const { return name; }
    int getAge() const { return age; }

    // Setter with validation
    void setName(std::string n) {
        if (!n.empty()) {
            name = n;
        } else {
            std::cout << "Error: name cannot be empty." << std::endl;
        }
    }

    void setAge(int a) {
        if (a >= 0 && a <= 150) {
            age = a;
        } else {
            std::cout << "Error: age " << a << " is out of range." << std::endl;
        }
    }

    void print() const {
        std::cout << name << ", age " << age << std::endl;
    }
};

Person p;
p.setName("Bob");
p.setAge(30);
p.print();

p.setAge(-5);   // rejected by setter
p.setAge(200);  // rejected by setter
p.print();      // age unchanged
// Expected output:
// Bob, age 30
// Error: age -5 is out of range.
// Error: age 200 is out of range.
// Bob, age 30

**Exercise 1:** Using the `BankAccount` class defined above, add a `withdraw` method that refuses to go below zero balance and prints an error message if the withdrawal would cause an overdraft. (The class already has a withdraw — enhance its validation or verify the current behavior with a test.)


In [ ]:
#include <iostream>
#include <string>

// Define an enhanced BankAccount class and test it here
// Your code here

## The this Pointer

Inside any non-static member function, `this` is a pointer to the current object. You rarely need to use it explicitly, but it's useful when:
1. A parameter name shadows a member name
2. You want to return the current object (for method chaining)


In [ ]:
#include <iostream>
#include <string>

class Counter {
private:
    int count;
    std::string label;

public:
    // 'count' and 'label' are parameter names — they shadow the members
    // Use this-> to disambiguate
    void init(int count, std::string label) {
        this->count = count;   // this->count is member, count is parameter
        this->label = label;
    }

    // Returning *this enables method chaining
    Counter &increment() {
        count++;
        return *this;  // return reference to the current object
    }

    Counter &add(int n) {
        count += n;
        return *this;
    }

    void print() const {
        std::cout << label << ": " << count << std::endl;
    }
};

Counter c;
c.init(0, "clicks");
c.increment().increment().add(5).increment();  // method chaining
c.print();
// Expected output:
// clicks: 7

## Scope Resolution ::

In real projects, class declarations go in header files (.h) and method definitions go in source files (.cpp). Outside the class body, you prefix the method name with `ClassName::` to tell the compiler which class it belongs to.


In [ ]:
#include <iostream>
#include <string>

// Class declaration (would normally be in a .h file)
class Logger {
private:
    std::string prefix;
    int messageCount;

public:
    void init(std::string p);     // declared here
    void log(std::string msg);    // declared here
    int getCount() const;         // declared here
};

// Method definitions (would normally be in a .cpp file)
// Note the ClassName:: prefix required outside the class body
void Logger::init(std::string p) {
    prefix = p;
    messageCount = 0;
}

void Logger::log(std::string msg) {
    messageCount++;
    std::cout << "[" << prefix << "] " << msg << std::endl;
}

int Logger::getCount() const {
    return messageCount;
}

Logger logger;
logger.init("APP");
logger.log("Starting up");
logger.log("Processing data");
logger.log("Done");
std::cout << "Total messages: " << logger.getCount() << std::endl;
// Expected output:
// [APP] Starting up
// [APP] Processing data
// [APP] Done
// Total messages: 3

## const Methods

A `const` method (trailing `const`) promises not to modify any member variables. Benefits:
- Documents intent: this method is read-only
- **Required** when calling on a `const` object
- The compiler enforces the promise

Rule of thumb: any method that only reads, never writes, should be marked `const`.


In [ ]:
#include <iostream>
#include <string>

class Circle {
private:
    double radius;

public:
    void setRadius(double r) { radius = r; }  // non-const: modifies object

    double getRadius() const { return radius; }           // const: read-only
    double area() const { return 3.14159 * radius * radius; }  // const: read-only
    double circumference() const { return 2 * 3.14159 * radius; }  // const

    void describe() const {
        std::cout << "Circle(r=" << radius
                  << ", area=" << area()
                  << ", circumference=" << circumference() << ")" << std::endl;
    }
};

Circle c;
c.setRadius(5.0);
c.describe();

// const object: can only call const methods
const Circle fixed = c;  // copy
// fixed.setRadius(10);  // ERROR: cannot modify const object
std::cout << "Fixed circle area: " << fixed.area() << std::endl;  // OK: const method
// Expected output:
// Circle(r=5, area=78.5397, circumference=31.4159)
// Fixed circle area: 78.5397

**Exercise 2:** Create a class `Temperature` with a private value stored in Celsius. Add const methods `toCelsius()`, `toFahrenheit()` (`F = C * 9/5 + 32`), and `toKelvin()` (`K = C + 273.15`). Test all three.


In [ ]:
#include <iostream>

// Your code here

## Encapsulation Benefits

Let's see a concrete example of how encapsulation allows changing the internal implementation without breaking external code.


In [ ]:
#include <iostream>
#include <string>

// Version 1 of FullName: stores as one string internally
class FullName {
private:
    // Internal representation: stored as "First Last"
    std::string fullName;

public:
    void set(std::string first, std::string last) {
        fullName = first + " " + last;
    }

    std::string getFirst() const {
        size_t spacePos = fullName.find(' ');
        if (spacePos != std::string::npos)
            return fullName.substr(0, spacePos);
        return fullName;
    }

    std::string getLast() const {
        size_t spacePos = fullName.find(' ');
        if (spacePos != std::string::npos)
            return fullName.substr(spacePos + 1);
        return "";
    }

    std::string getFull() const { return fullName; }
};

// Caller code — doesn't know how it's stored internally
FullName fn;
fn.set("John", "Doe");
std::cout << "First: " << fn.getFirst() << std::endl;
std::cout << "Last: "  << fn.getLast()  << std::endl;
std::cout << "Full: "  << fn.getFull()  << std::endl;
// If we later change to store first/last as separate strings,
// the caller code above stays IDENTICAL.
// Expected output:
// First: John
// Last: Doe
// Full: John Doe

## Final Exercise

Design a class `Student` with:
- Private members: `name` (std::string), `grade` (int 0-100)
- A method `init(name, grade)` that initializes the fields (with validation for grade range)
- A getter for name, a getter for grade
- A setter for grade that validates the range 0-100
- A private helper method `getLetterGrade() const` that converts the numeric grade to a letter (A: 90-100, B: 80-89, C: 70-79, D: 60-69, F: below 60)
- A public method `printReport() const` that prints the student's name, numeric grade, and letter grade

Test with at least two students and try setting an invalid grade.


In [ ]:
#include <iostream>
#include <string>

// Your code here

## Modern C++ (C++11 and Beyond)


In [ ]:
#include <iostream>

// C++11: in-class member initialization
// Members can have default values right in the class body
class Config {
public:
    int maxRetries = 3;          // default value, C++11
    double timeout = 30.0;       // default value, C++11
    bool verboseMode = false;    // default value, C++11

    void print() const {
        std::cout << "maxRetries=" << maxRetries
                  << ", timeout=" << timeout
                  << ", verbose=" << (verboseMode ? "yes" : "no")
                  << std::endl;
    }
};

Config cfg;       // all defaults
cfg.print();

cfg.maxRetries = 5;
cfg.verboseMode = true;
cfg.print();
// Expected output:
// maxRetries=3, timeout=30, verbose=no
// maxRetries=5, timeout=30, verbose=yes

In [ ]:
#include <iostream>

// C++11: strongly-typed enums (enum class)
// Old C enums: values leak into surrounding scope, can be compared to int
// enum Direction { North, South, East, West };  // North is in global scope

// C++11 enum class: scoped and type-safe
enum class Direction { North, South, East, West };
enum class Color { Red, Green, Blue };

Direction d = Direction::North;  // must be fully qualified
Color c = Color::Red;

// d == c;  // ERROR: cannot compare Direction and Color (type-safe!)
// d == 0;  // ERROR: cannot compare enum class to int

if (d == Direction::North) {
    std::cout << "Heading North" << std::endl;
}

// Can switch on enum class
switch (c) {
    case Color::Red:   std::cout << "Color: Red"   << std::endl; break;
    case Color::Green: std::cout << "Color: Green" << std::endl; break;
    case Color::Blue:  std::cout << "Color: Blue"  << std::endl; break;
}
// Expected output:
// Heading North
// Color: Red